# Initial Setup

In [0]:
%pip install catboost lightgbm optuna holidays duckdb

## Imports

In [0]:
import numpy as np
import pandas as pd
import duckdb

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
%matplotlib inline

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, TimeSeriesSplit, cross_val_predict, cross_val_score, cross_validate
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.base import clone
from sklearn.compose import ColumnTransformer

from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool

import optuna
from typing import Any
import warnings

import holidays

## Variables

In [0]:
def check_environment():
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

ENVIRONMENT = check_environment()

DRIVE_FOLDER_NAME = "Store Sales"

In [0]:
import plotly.io as pio
pio.renderers.default = "vscode"

In [0]:
TARGET = "sales"
N_SPLITS = 5
SEED = 42
eps = 1e-5

def check_environment():
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

ENVIRONMENT = check_environment()

non_relevant_columns = ['id', 'source']


## Helper Functions

### Data Loader

In [0]:
def load_data(option: str="local"):
  '''
  returns train_df, test_df, df

  '''
  if option == "local":
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    train_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/train.csv')
    test_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/test.csv')

  train_df['source'] = 'train'
  test_df['source'] = 'test'
  train_df['date'] =  pd.to_datetime(train_df['date'])
  test_df['date'] =  pd.to_datetime(test_df['date'])
  df = pd.concat([train_df, test_df], ignore_index=True)

  return train_df, test_df, df

### Loader of additional data

In [0]:
def load_helper_data(option: str="local"):
  '''
  returns oil_df, transactions_df, holidays_events_df, stores_df

  '''
  if option == "local":
    oil_df = pd.read_csv('oil.csv')
    transactions_df = pd.read_csv('transactions.csv')
    holidays_events_df = pd.read_csv('holidays_events.csv')
    stores_df = pd.read_csv('stores.csv')


  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    oil_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/oil.csv')
    transactions_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/transactions.csv')
    holidays_events_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/holidays_events.csv')
    stores_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/stores.csv')
    
  oil_df['date'] =  pd.to_datetime(oil_df['date'])
  transactions_df['date'] =  pd.to_datetime(transactions_df['date'])
  holidays_events_df['date'] =  pd.to_datetime(holidays_events_df['date'])
  


  return oil_df, transactions_df, holidays_events_df, stores_df

### Data Splitter

In [0]:
def data_splitter(df: pd.DataFrame, additional_drop_columns: list = []) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    '''
    return: X_train, y_train, submission_df

    '''
    train_dataset = df[df['source'] == 'train'].drop(columns=non_relevant_columns + additional_drop_columns)
    submission_dataset = df[df['source'] == 'test'].drop(columns=['source', TARGET] + additional_drop_columns)

    X_train = train_dataset.drop(columns=[TARGET])
    y_train = train_dataset[TARGET]

    return X_train, y_train, submission_dataset

## Create DataFrames

In [0]:
train_df, test_df, df = load_data(ENVIRONMENT)
oil_df, transactions_df, holidays_events_df, stores_df = load_helper_data(ENVIRONMENT)

# Tables infos

## Overall Info

### Main DF

In [0]:
df.head()

In [0]:
df.groupby('source').agg(min_date=('date', 'min'), max_date=('date', 'max'))

In [0]:
df.describe(include='all')

### Stores

In [0]:
stores_df.head()

In [0]:
stores_df.describe(include='all')

### Transactions

In [0]:
transactions_df.head()

In [0]:
transactions_df.describe(include='all')

### Holidays

In [0]:
holidays_events_df.head()

In [0]:
holidays_events_df.describe()

### Oil

In [0]:
oil_df.head()

In [0]:
oil_df.describe()

In [0]:
df.head()

# Merging datasets

In [0]:
is_transferred = holidays_events_df['transferred'] == True
legit_holidays = holidays_events_df['type'].isin(['Holiday', 'Transfer', 'Bridge', 'Additional'])
is_workday = holidays_events_df['type']  == 'Work Day'
is_event = holidays_events_df['type']  == 'Event'

is_national = holidays_events_df['locale'] == 'National'
is_regional = holidays_events_df['locale'] == 'Regional'
is_local = holidays_events_df['locale'] == 'Local'

national_holidays = holidays_events_df.loc[(~is_transferred) & (is_national) & (legit_holidays)]
regional_holidays = holidays_events_df.loc[(~is_transferred) & (is_regional) & (legit_holidays)]
local_holidays = holidays_events_df.loc[(~is_transferred) & (is_local) & (legit_holidays)]

In [0]:
is_transferred = holidays_events_df['transferred'] == True
legit_holidays = holidays_events_df['type'].isin(['Holiday', 'Transfer', 'Bridge', 'Additional'])
is_workday = holidays_events_df['type']  == 'Work Day'
is_event = holidays_events_df['type']  == 'Event'

is_national = holidays_events_df['locale'] == 'National'
is_regional = holidays_events_df['locale'] == 'Regional'
is_local = holidays_events_df['locale'] == 'Local'

national_holidays = holidays_events_df.loc[(~is_transferred) & (is_national) & (legit_holidays)]
regional_holidays = holidays_events_df.loc[(~is_transferred) & (is_regional) & (legit_holidays)]
local_holidays = holidays_events_df.loc[(~is_transferred) & (is_local) & (legit_holidays)]

national_holidays = national_holidays.groupby('date').agg(nat_num_holidays=('type', 'count'), nat_holiday_types =('type', ', '.join)).sort_values(by='nat_num_holidays', ascending=False).reset_index()
national_holidays['is_national'] = 1

regional_holidays = regional_holidays.groupby(['date', 'locale_name']).agg(reg_num_holidays=('type', 'count'), reg_holiday_types =('type', ', '.join)).sort_values(by='reg_num_holidays', ascending=False).reset_index()
regional_holidays['is_regional'] = 1

local_holidays = local_holidays.groupby(['date', 'locale_name']).agg(loc_num_holidays=('type', 'count'), local_holiday_types =('type', ', '.join)).sort_values(by='loc_num_holidays', ascending=False).reset_index()
local_holidays['is_local'] = 1


workdays = holidays_events_df.loc[(~is_transferred) & (is_workday)].copy()
workdays['is_workday'] = 1
workdays = workdays[['date', 'is_workday']].drop_duplicates(subset=['date'], keep='first')

events = holidays_events_df.loc[(~is_transferred) & (is_event)].copy()
events['is_event'] = 1
events = events[['date', 'is_event']].drop_duplicates(subset=['date'], keep='first')

In [0]:
initial_length = len(df)

oil_df = (oil_df.set_index("date")
                .reindex(pd.date_range(oil_df["date"].min(), df["date"].max()))
                .ffill().bfill()
                .rename_axis("date").reset_index())

df = (df.merge(transactions_df, on=['date', 'store_nbr'], how='left')
            .merge(stores_df, on='store_nbr', how='left')
            .merge(oil_df, on='date', how='left')
            .merge(national_holidays, on='date', how='left')
            .merge(regional_holidays, left_on=['date', 'state'], right_on=['date','locale_name'], how='left').drop(columns=['locale_name'])
            .merge(local_holidays, left_on=['date','city'], right_on=['date','locale_name'], how='left').drop(columns=['locale_name'])
            .merge(workdays, on='date', how='left')
						.merge(events, on='date', how='left')
            )

In [0]:
final_length = len(df)

In [0]:
assert initial_length == final_length, 'Lenght is not matching'

In [0]:
df.loc[df['source'] == 'train'].isna().sum()

## Null Adjustments

In [0]:
is_train = df["sales"].notna()

df["store_sales"] = df.groupby(["date", "store_nbr"])["sales"].transform("sum")

transaction_null = df["transactions"].isna()
store_closed = df["store_sales"] == 0

df.loc[is_train & transaction_null & store_closed, "transactions"] = 0

In [0]:
num_holiday_cols = ['is_national', 'is_regional', 'is_local', 'is_workday', 'is_event', 'nat_num_holidays', 'reg_num_holidays', 'loc_num_holidays']
str_holiday_cols = ['nat_holiday_types', 'reg_holiday_types', 'local_holiday_types']

df[num_holiday_cols] = df[num_holiday_cols].fillna(0).astype(int)
df[str_holiday_cols] = df[str_holiday_cols].fillna('none')

In [0]:
df.loc[df['source'] == 'train'].isna().sum()

# EDA

In [0]:
df.info()

In [0]:
df.describe(include='object').T

In [0]:
df.describe(include='number').T

In [0]:
df.columns

## Store Nbr

In [0]:
plt.figure(figsize=(15,5))
ax1 = sns.barplot(
    df,
    x='store_nbr',
    y='store_sales',
    estimator='sum',
    errorbar=None,
)

ax2 = ax1.twinx()

sns.pointplot(
    data=df,
    x='store_nbr',
    y='store_sales',
    estimator='mean',
    ax=ax2,
    color='red',
    markers='x',
    linestyles='-',
    label="Mean Daily store_sales",
)


sns.pointplot(
    data=df,
    x='store_nbr',
    y='store_sales',
    estimator='median',
    ax=ax2,
    color='orange',
    markers='*',
    linestyles='--',
    label = "Median Daily store_sales",
)

In [0]:
col, metric = ["sales", "store_sales"], ["sum", "median"]

fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(16, 10))
for i, c in enumerate(col):         # i -> row
    for j, m in enumerate(metric): # j - column
        order = df.groupby("store_nbr")[c].agg(m).sort_values().index
        plot_ax = axs[i,j]
        sns.barplot(
            df,
            x="store_nbr",
            y=c,
            estimator=m,
            errorbar=None,
            ax=plot_ax,
            order=order,
            hue='type'
            
        )
        plot_ax.set_title(f"{m.capitalize()} of {c.capitalize()} per store")
        plot_ax.tick_params(axis="x", rotation=270)
plt.tight_layout()

In [0]:
stats = df.groupby("store_nbr")['sales'].agg(median = 'median').reset_index()
zero_median = stats['median'] == 0
stats.loc[zero_median, 'median_sales'] = 'Zero'

stats.loc[~zero_median,'median_sales'] = pd.qcut(
        stats.loc[~zero_median,'median'],
        q=6,
        labels=['Low', 'Lower-mid', 'Mid', 'High', 'Top',]
)

stats.sort_values(by='median')

In [0]:
col, metric = ["sales", "transaction"], ["sum", "median"]

fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(16, 10))
for i, c in enumerate(col):         # i -> row
    for j, m in enumerate(metric): # j - column
        order = df.groupby("family")[c].agg(m).sort_values().index
        plot_ax = axs[i,j]
        sns.barplot(
            df,
            x="family",
            y=c,
            estimator=m,
            errorbar=None,
            ax=plot_ax,
            order=order,
            
        )
        plot_ax.set_title(f"{m.capitalize()} of {c.capitalize()} per store")
        plot_ax.tick_params(axis="x", rotation=270)
plt.tight_layout()

In [0]:
plt.figure(figsize=(15,5))
ax1 = sns.barplot(
    df,
    x='store_nbr',
    y='sales',
    estimator='sum',
    errorbar=None,
    order=
)

In [0]:
store_family_summary = df.groupby(["store_nbr", "family"]).agg(
    sum_sales=("sales", "sum"),
    median_sales=("sales", "median"),
    average_sales=("sales", "mean"),
).reset_index().pivot(
    index='family',
    columns='store_nbr',
    values=['sum_sales', 'median_sales', 'average_sales']
)

In [0]:
order = store_family_summary['sum_sales'].sum(axis=1).sort_values(ascending=False).index
store_family_summary = store_family_summary['sum_sales'].loc[order]

In [0]:
store_family_summary.head()

In [0]:
plt.figure(figsize=(15,12))
ax = sns.heatmap(
    store_family_summary/ 10000,
    # annot=True,
    fmt=f",.0f",
    cmap="YlGnBu",
    linewidths=0.2,
    annot_kws={"size": 7},
    square=True,
    robust=True,
    cbar_kws={"shrink": 0.4, "aspect": 30, "pad": 0.01},  
)

ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
plt.tight_layout()

In [0]:
print(1)